In [1]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
data = pd.read_csv('imdb_dataset.csv')


In [3]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [6]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

data['sentiment'] = encoder.fit_transform(data['sentiment'])

In [7]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english')) - {'not','no','very','too','just'}

def pipeline(text):
    # 1. Cleaning: HTML, URLs, Mentions, Hashtags, Non-alphabetic
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[@#]\w+', ' ', text)
    text = re.sub(r"[^a-zA-Z\s.!?']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # 2. Tokenization & Lowercasing
    text = text.lower()
    word_tokens = word_tokenize(text)

    # 3. Lemmatization & Stopword Removal
    # We keep the words in a list to be joined back into a clean string
    cleaned_tokens = [
        lemmatizer.lemmatize(word, pos='v') 
        for word in word_tokens 
        if word not in stop_words
    ]

    return " ".join(cleaned_tokens)


#Runing the pipeline on every document
data['review'] = data['review'].apply(pipeline)

# Step 2: Apply TfidfVectorizer (This handles the N-grams automatically)
tfidf = TfidfVectorizer(
    ngram_range=(1,3),     
    max_features=5000,
    min_df=5,
    max_df=0.8,
    sublinear_tf=True
)



In [42]:
print(list(tfidf.vocabulary_.keys())[:20])  # Show the first 20 features to verify

['one', 'reviewers', 'mention', 'watch', 'just', 'oz', 'episode', 'll', 'hook', 'right', 'exactly', 'happen', 'first', 'thing', 'strike', 'scenes', 'violence', 'set', 'word', 'go']


In [45]:
print(X[0].toarray())  # Show the TF-IDF vector for the first document to verify

[[0. 0. 0. ... 0. 0. 0.]]


In [8]:
X = tfidf.fit_transform(data['review'])
y = data['sentiment']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
from sklearn.linear_model import LogisticRegression 
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import classification_report, confusion_matrix , precision_score


In [11]:

model = LogisticRegression(solver='liblinear')
model.fit(X_train, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [12]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print('========================================================================')
print('CONFUSION MATRIX :\n',confusion_matrix(y_test, y_pred))
print('========================================================================')
print("Precision Score:", precision_score(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.89      0.91      0.90      5039

    accuracy                           0.89     10000
   macro avg       0.90      0.89      0.89     10000
weighted avg       0.90      0.89      0.89     10000

CONFUSION MATRIX :
 [[4376  585]
 [ 467 4572]]
Precision Score: 0.8865619546247818


# Error analysis Report 

In [13]:
# Use the test set rows (same length as y_test/y_pred), not the full X (50000 rows)
results = pd.DataFrame({
    'text': data.loc[y_test.index, 'review'].values,
    'actual': y_test.values,
    'predicted': y_pred
})
errors = results[results['actual'] != results['predicted']]

false_positives = errors[(errors['actual'] == 0) & (errors['predicted'] == 1)]
false_negatives = errors[(errors['actual'] == 1) & (errors['predicted'] == 0)]



In [14]:
X.shape 

(50000, 5000)

# Emotion Detection


In [33]:
emotion_dict = {
    "joy": ["happy", "great", "amazing", "love", "fantastic", "awesome", "fun", "enjoyed", "brilliant", "excellent", "wonderful", "delightful", "pleased", "satisfied", "positive", "enchanting", "charming", "glad"],
    
    "anger": ["hate", "worst", "terrible", "angry", "bad", "annoying", "frustrating" , "disgusting", "horrible", "pathetic", "ridiculous", "awful", "annoying", "frustrating", "dull", "disappointing", "slow", "waste", "poor"],
    
    "sadness": ["sad", "boring", "dull", "disappointing", "slow", "waste", "poor" ,     "depressing", "heartbreaking", "tragic", "melancholy", "gloomy", "miserable", "lonely", "hopeless", "dull", "disappointing", "slow", "waste", "poor"],
    
    "surprise": ["unexpected", "shocking", "surprising", "twist", "unpredictable", "astonishing", "startling", "amazing", "stunning", "breathtaking", "unexpected", "shocking", "surprising", "twist", "unpredictable", "astonishing", "startling", "amazing", "stunning", "breathtaking"],
    
    "disgust": ["awful", "disgusting", "horrible", "pathetic", "ridiculous" , "annoying", "frustrating", "dull", "disappointing", "slow", "waste", "poor", "boring", "dull", "disappointing", "slow", "waste", "poor"]
}


In [34]:
def get_emotion_score(text , emotion_dict):
    tokens = text.split()

    emotion_score = {key : 0 for key in emotion_dict}

    for word in tokens:
        for emotion , words in emotion_dict.items():
            if word in words:
                emotion_score[emotion] += 1
    # Normalize 
    total = sum(emotion_score.values())

    if total > 0:
        for key in emotion_score:
            emotion_score[key] = emotion_score[key]/total
    return emotion_score

In [35]:
data['emotion'] = data['review'].apply(lambda x: get_emotion_score(x , emotion_dict))


In [38]:
data['emotion'][14]

{'joy': 0.5, 'anger': 0.5, 'sadness': 0.0, 'surprise': 0.0, 'disgust': 0.0}

# POS Tagging

In [48]:
def get_writing_style(text):
    tokens = text.split()
    
    pos_tags = nltk.pos_tag(tokens)
    total_words = len(tokens)
    noun_count = 0
    verb_count = 0  
    adj_count = 0
    adv_count = 0

    for word ,tag in pos_tags:
        if tag.startswith('NN'):
            noun_count+=1
        elif  tag.startswith('VB'):
            verb_count+=1
        elif tag.startswith('JJ'):
            adj_count+=1
        elif tag.startswith('RB'):
            adv_count+=1

    # avoid division by zero
    if total_words == 0:
        return{}
    
    return {
        "noun_ratio": noun_count/total_words,
        "verb_ratio": verb_count / total_words,
        "adj_ratio": adj_count / total_words,
        "adv_ratio": adv_count / total_words,
        "avg_word_length": sum(len(w) for w in tokens) / total_words,
        "sentence_length": total_words
    }


In [50]:
import nltk

nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')

data['style'] = data['review'].apply(get_writing_style)

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/kushal09/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/kushal09/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt to /Users/kushal09/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [52]:
data.head()

,review,sentiment,emotion,style
0,one reviewers mention watch just oz episode 'l...,1,"{'joy': 0, 'anger': 0, 'sadness': 0, 'surprise...","{'noun_ratio': 0.40932642487046633, 'verb_rati..."
1,wonderful little production . film technique v...,1,"{'joy': 1.0, 'anger': 0.0, 'sadness': 0.0, 'su...","{'noun_ratio': 0.3627450980392157, 'verb_ratio..."
2,think wonderful way spend time too hot summer ...,1,"{'joy': 1.0, 'anger': 0.0, 'sadness': 0.0, 'su...","{'noun_ratio': 0.422680412371134, 'verb_ratio'..."
3,basically 's family little boy jake think 's z...,0,"{'joy': 0, 'anger': 0, 'sadness': 0, 'surprise...","{'noun_ratio': 0.4430379746835443, 'verb_ratio..."
4,petter mattei 's love time money visually stun...,1,"{'joy': 1.0, 'anger': 0.0, 'sadness': 0.0, 'su...","{'noun_ratio': 0.49295774647887325, 'verb_rati..."


In [57]:
print(data['style'][0])

{'noun_ratio': 0.40932642487046633, 'verb_ratio': 0.16580310880829016, 'adj_ratio': 0.19170984455958548, 'adv_ratio': 0.10880829015544041, 'avg_word_length': 5.005181347150259, 'sentence_length': 193}
